# Example run of inverse problem.

## Preparing the notebook.

### Flags for autoreloading.

In [ ]:
AUTORELOAD = True
IS_EXT_LOADED = False

### Optionally autoreload notebook.

In [ ]:
if AUTORELOAD:
    if not IS_EXT_LOADED:
        %load_ext autoreload
        IS_EXT_LOADED = True
    assert IS_EXT_LOADED
    %autoreload 2

### Importing the libraries.

In [ ]:
import sys
from typing import Literal, Type

import cloudpickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.spatial.distance import cdist
from scipy.special import softmax
from scipy.stats import gaussian_kde
from sklearn.neighbors import KernelDensity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import torch
import yaml

from sc_exp_design.constants import DataFields, ParamsFields, PredictionFields
from sc_exp_design.config import NeuralVelocityFieldConfig
from sc_exp_design.models import FlowMatching, TargetPredictionModel
from sc_exp_design.utils import set_reproducibility
from sc_exp_design.training.callbacks import MetricsCallBack, TrainingCallBacks


### Constants.

In [ ]:
BASE_DIR = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC"
# PERTURBATION_RESPONSE_MODEL_CHECKPOINT_PATH = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/dumps/2025-11-14_15-15-26/lively-smoke-29_FlowMatching.pkl"
# PERTURBATION_RESPONSE_MODEL_CHECKPOINT_PATH = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/dumps/2025-11-14_15-15-26/sage-energy-324_FlowMatching.pkl"
# PERTURBATION_RESPONSE_MODEL_CHECKPOINT_PATH = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/dumps/2025-11-14_15-15-26/winter-butterfly-1006_FlowMatching.pkl"
# PERTURBATION_RESPONSE_MODEL_CHECKPOINT_PATH = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/dumps/2025-12-14_11-53-04/glowing-plasma-8_FlowMatching.pkl" # in distribution
# PERTURBATION_RESPONSE_MODEL_CHECKPOINT_PATH = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/dumps/2025-12-14_11-53-04/cerulean-durian-19_FlowMatching.pkl" # in distribution
PERTURBATION_RESPONSE_MODEL_CHECKPOINT_PATH = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/dumps/2026-02-02_19-09-41/charmed-puddle-158_FlowMatching.pkl" # in distribution

TARGET_PREDICTION_MODEL_CHECKPOINT_PATH = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/dumps/2025-11-14_21-22-45/dashing-frog-1258_TargetPredictionModel.pkl"
ANNOTATION_CONFIG_PATH = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC/generative_modeling/conditional_flow_matching/config/annotation/default.yaml"

perturbation_obs_key = "protocol_id"
cell_state_rep = "X_channel_standardized+X_scatter_standardized"

RANDOM_SEED = 42

### Importing Additional Modules.

In [ ]:
sys.path.insert(0, f"{BASE_DIR}/inverse/inverse_utils")
sys.path.insert(0, f"{BASE_DIR}/model_utils")
from forward_model import ForwardModel
from lambda_schedulers import ExponentialDecayScheduler
from loss_guidance import LossGuidedFlow
from z_norm_modules import ZNorm, IZNorm, RescaledTargetPredictionModel


### Reading annotation config.

In [ ]:
with open(ANNOTATION_CONFIG_PATH, "r") as fb:
    annotation_dict = yaml.safe_load(fb)

### Retrieving the number of scatter columns.

In [ ]:
scatter_columns = annotation_dict["scatter_columns"]
n_scatter_feats = len(scatter_columns)
print(f"{n_scatter_feats=}")

### Setting seed for reproducibility.

In [ ]:
set_reproducibility(RANDOM_SEED)

### Utility function for predictions.

In [ ]:
def get_prediction_adata(
    idx,
    sample,
    X_channel,
    X_scatter,
    train_adata_fwd,
    val_adata_fwd,
    perturbation_id,
    nn_val,
    perturbation_obs_key,
    cell_state_rep,
    perturbation_reps,
):
    # concatenating with predictions
    x_channel = np.concatenate(
        (
            train_adata_fwd.obsm[cell_state_rep][:, :-n_scatter_feats],
            val_adata_fwd.obsm[cell_state_rep][:, :-n_scatter_feats],
            X_channel,
        ), axis=0
    )
    x_scatter = np.concatenate(
        (
            train_adata_fwd.obsm[cell_state_rep][:, -n_scatter_feats:],
            val_adata_fwd.obsm[cell_state_rep][:, -n_scatter_feats:],
            X_scatter,
        ), axis=0
    )

    # handling conditions
    conditions = np.concatenate(
        (
            train_adata_fwd.obsm[perturbation_id],
            val_adata_fwd.obsm[perturbation_id],
            np.repeat(sample[None], X_channel.shape[0], axis=0)
        )
    )
    unique_perts = np.unique(conditions, axis=0)
    perts_id = np.apply_along_axis(
        lambda e: np.where(
            np.all(
                unique_perts == e, axis=-1
            ))[0].item(),
        -1,
        conditions
    ).tolist()

    # retrieving the mask for current nearest neighbor
    train_cond_mask = np.all(np.isclose(train_adata_fwd.obsm[perturbation_id], nn_val, atol=1e-5, rtol=1e-3), axis=-1)

    # preparing the annotations
    obs = pd.DataFrame(
        {
            "dataset_type": [
                "in-distribution" for _ in range(len(train_adata_fwd))
            ] + [
                "ood" for _ in range(len(val_adata_fwd))
            ] + [
                "generated" for _ in range(len(X_channel))
            ],
            "is_generated": [
                False for _ in range(len(train_adata_fwd))
            ] + [
                False for _ in range(len(val_adata_fwd))
            ] + [
                True for _ in range(len(X_channel))
            ],
            "is_true_ood": [
                False for _ in range(len(train_adata_fwd))
            ] + [
                True for _ in range(len(val_adata_fwd))
            ] + [
                False for _ in range(len(X_channel))
            ],
            "is_true_nn_cond": train_cond_mask.tolist() + [
                False for _ in range(len(val_adata_fwd))
            ] + [
                False for _ in range(len(X_channel))
            ],
            perturbation_obs_key: perts_id
        }
    )
    obsm = {
        "X_scatter": x_scatter,
        perturbation_reps: conditions,
    }
    uns = {"sampled_pert": sample, "nn_vals": nn_val}
    var = pd.DataFrame(index=train_adata_fwd.var_names)

    # creating adata
    pred_adata_group = sc.AnnData(
        X=x_channel,
        obs=obs,
        obsm=obsm,
        var=var,
        uns=uns
    )

    # computing pcs, neighbors, umap
    sc.pp.pca(pred_adata_group)
    sc.pp.neighbors(pred_adata_group)
    sc.tl.umap(pred_adata_group)
    
    # storing the results
    print(f"AnnData created for prior sample {idx} -> {pred_adata_group}")
    return pred_adata_group


def generated_density_knn(X_true, X_generated, k=20, bandwidth=None):

    nbrs = NearestNeighbors(n_neighbors=k).fit(X_true)
    distances, indices = nbrs.kneighbors(X_generated)
    
    # Auto-bandwidth: median distance of neighbors
    if bandwidth is None:
        bandwidth = np.median(distances)

    density = np.zeros(X_true.shape[0])

    # Gaussian kernel weights
    weights = np.exp(-(distances**2) / (2 * bandwidth**2))

    # Add weighted contributions
    for gen_idx in range(len(X_generated)):
        for neighbor_pos, real_idx in enumerate(indices[gen_idx]):
            density[real_idx] += weights[gen_idx, neighbor_pos]

    return density

## Loading Pre-Trained Forward Model.

### Cellular Response Prediction Model.

In [ ]:
perturbation_response_prediction_model = FlowMatching.load(
    PERTURBATION_RESPONSE_MODEL_CHECKPOINT_PATH
)
perturbation_response_prediction_model.trainer.plot_training_logs()

### Retrieving z-standardization parameters for train data.

In [ ]:
X_channel_params_inv = perturbation_response_prediction_model.train_data.adata.uns["X_channel_params"]
X_scatter_params_inv = perturbation_response_prediction_model.train_data.adata.uns["X_scatter_params"]
print(f"{X_channel_params_inv.keys()=}, {X_scatter_params_inv.keys()=}")
params_inv = {
    ParamsFields.MEAN: torch.from_numpy(
        np.concatenate(
            (
                X_channel_params_inv[ParamsFields.MEAN],
                X_scatter_params_inv[ParamsFields.MEAN]
            ), axis=0
        )
    ),
    ParamsFields.COVARIANCE: torch.from_numpy(
        np.concatenate(
            (
                X_channel_params_inv["std"],
                X_scatter_params_inv["std"]
            ), axis=0
        )
    ),
}
perturbation_reps = next(iter(perturbation_response_prediction_model.train_data.data.perturbation_covariates))
izn = IZNorm(params_inv)
izn

### Target Prediction Model.

In [ ]:
target_prediction_model = TargetPredictionModel.load(
    TARGET_PREDICTION_MODEL_CHECKPOINT_PATH
)
ct_le = LabelEncoder().fit(target_prediction_model.train_data.adata.obs["cell_type"].values)
target_prediction_model.target_predictor_trainer.plot_training_logs()

### Retrieving z-standardization parameters for train data.

In [ ]:
X_channel_params_fwd = target_prediction_model.train_data.adata.uns["X_channel_params"]
X_scatter_params_fwd = target_prediction_model.train_data.adata.uns["X_scatter_params"]
print(f"{X_channel_params_fwd.keys()}, {X_scatter_params_fwd.keys()}")
params_fwd = {
    ParamsFields.MEAN: torch.from_numpy(
        np.concatenate(
            (
                X_channel_params_fwd[ParamsFields.MEAN],
                X_scatter_params_fwd[ParamsFields.MEAN]
            ), axis=0
        )
    ),
    ParamsFields.COVARIANCE: torch.from_numpy(
        np.concatenate(
            (
                X_channel_params_fwd["std"],
                X_scatter_params_fwd["std"]
            ), axis=0
        )
    ),
}
zn = ZNorm(params_fwd)
zn

### Initializing rescaled target prediction model.

In [ ]:
g_model = RescaledTargetPredictionModel(
    target_prediction_model.target_prediction_model,
    inv_params=izn,
    fwd_params=zn
)
le_ct = LabelEncoder().fit(target_prediction_model.train_data.adata.obs["cell_type"])

### Initializing forward model.

In [ ]:
forward_model = ForwardModel(
    forward_model=perturbation_response_prediction_model,
    target_prediction_model=g_model,
)

## Training Prior Flow.

### Retrieving the condition data.

In [ ]:
val_perturbation_data = next(iter(next(iter(perturbation_response_prediction_model.validation_data.values())).perturbation_data.values()))
train_perturbation_data = next(iter(perturbation_response_prediction_model.train_data.perturbation_data.values()))
print(f"{train_perturbation_data.shape=}, {val_perturbation_data.shape=}")

### Preparing train and validation adata.

In [ ]:
train_adata = sc.AnnData(
    X=train_perturbation_data,
    var=pd.DataFrame(index=annotation_dict["protocol_columns"])
)
val_adata = sc.AnnData(
    X=val_perturbation_data,
    var=pd.DataFrame(index=annotation_dict["protocol_columns"])
)
print(f"{train_adata=}, {val_adata=}")

### Initializing the model.

In [ ]:
prior_flow = FlowMatching(
    flow_type="rectified",
    coupling_type="independent",
    generate_from_noise=True
)
prior_flow

### Preparing prior flow data.

In [ ]:
prior_flow.prepare_train_data(train_adata)
prior_flow.prepare_validation_data("validation", val_adata)

### Preparing velocity field configuration.

In [ ]:
vf_config = NeuralVelocityFieldConfig(
    train_adata.X.shape[-1],
    state_encoder_mlp_kwargs={
        "hidden_dims": (64, ),
        "use_batchnorm": False,
        "use_dropout": False,
        "activation_class": torch.nn.ReLU,
        "final_activation_class": torch.nn.Identity,
    },
    state_encoder_output_dim=16,

    encode_time=True,
    use_sinusoidal_time_features=True,
    time_features_num_freqs=64,
    time_encoder_mlp_kwargs={
        "hidden_dims": (32, ),
        "use_batchnorm": False,
        "use_dropout": False,
        "activation_class": torch.nn.ReLU,
        "final_activation_class": torch.nn.Identity,
    },
    time_encoder_output_dim=16,
    use_guidance=False,
    decoder_mlp_kwargs={
        "hidden_dims": (128, 128,),
        "use_batchnorm": False,
        "use_dropout": False,
        "activation_class": torch.nn.ReLU,
        "final_activation_class": torch.nn.Identity,
    },
    conditioning_type="resnet"
)


### Initializing model.

In [ ]:
optimizer_kwargs = {"lr": 1e-4}
prior_flow.prepare_model(
    vf_config,
    optimizer_class=torch.optim.Adam,
    optimizer_kwargs=optimizer_kwargs
)

### Set up callbacks.

In [ ]:
metrics_cb = MetricsCallBack(["energy_distance", "r_squared"])
cb = TrainingCallBacks([metrics_cb])

### Training the model.

In [ ]:
prior_flow.train(
    num_training_steps=100_000,
    valid_freq=50_000,
    train_batch_size=4096,
    validation_batch_size=10_000,
    num_grad_accumulation_steps=100,
    callbacks=cb
)
prior_flow.save("../", model_prefix="prior_flow")


### Plot training logs.

In [ ]:
keys_to_plot = list(prior_flow.trainer.training_logs.keys())
prior_flow.trainer.plot_training_logs(
    figsize=(4*len(keys_to_plot), 4),
    keys_to_plot=keys_to_plot
)


## Prepare base adata.

### Subsampling train and validation data.

In [ ]:
n_train_cells = 75_000
n_val_cells = 25_000
train_adata_fwd = sc.pp.sample(
    perturbation_response_prediction_model.train_data.adata,
    n=n_train_cells,
    copy=True
)
val_adata_fwd = sc.pp.sample(
    next(iter(perturbation_response_prediction_model.validation_data.values())).adata,
    n=n_val_cells,
    copy=True
)
print(f"{train_adata_fwd=}, {val_adata_fwd=}")

### Predicting cell type on train and validation data.

In [ ]:
X_train = train_adata_fwd.obsm[cell_state_rep]
X_val = val_adata_fwd.obsm[cell_state_rep]
g_model.eval()
with torch.no_grad():
    G_logits_train = g_model(
        torch.from_numpy(X_train).float().to("cuda")
    )["cell_type"].detach().cpu().numpy()
    G_logits_val = g_model(
        torch.from_numpy(X_val).float().to("cuda")
    )["cell_type"].detach().cpu().numpy()
G_probs_train = softmax(G_logits_train, axis=1)
G_probs_val = softmax(G_logits_val, axis=1)
G_id_train = G_probs_train.argmax(1)
G_id_val = G_probs_val.argmax(1)
G_label_train = ct_le.inverse_transform(G_id_train)
G_label_val = ct_le.inverse_transform(G_id_val)
target_probs = G_probs_val.mean(0)
print(f"{G_logits_train.shape=}, {G_logits_val.shape=}, {G_probs_train.shape=}, {G_probs_val.shape=}, {G_id_train.shape=}, {G_id_val.shape=}, {G_label_train.shape=}, {G_label_val.shape=}")

### Constructing base adata.

In [ ]:
X_channel_train = X_train[:, :-n_scatter_feats]
X_scatter_train = X_train[:, -n_scatter_feats:]
X_channel_val = X_val[:, :-n_scatter_feats]
X_scatter_val = X_val[:, -n_scatter_feats:]

X_channel_joint = np.concatenate((X_channel_train, X_channel_val), axis=0)
X_scatter_joint = np.concatenate((X_scatter_train, X_scatter_val), axis=0)

obs = pd.DataFrame({
    "cell_type": np.concatenate((G_label_train, G_label_val), axis=0),
    "dataset_type": ["train"]*G_label_train.shape[0] + ["val"]*G_label_val.shape[0]
})
obsm = {
    "X_scatter": X_scatter_joint,
    "cell_type_logits": np.concatenate((G_logits_train, G_logits_val), axis=0),
    cell_state_rep: np.concatenate((X_train, X_val), axis=0),
    annotation_dict["protocol_obsm_key"]: np.concatenate(
        (
            train_adata_fwd.obsm[annotation_dict["protocol_obsm_key"]],
            val_adata_fwd.obsm[annotation_dict["protocol_obsm_key"]]
        ), axis=0
    )   
}

adata_base = sc.AnnData(
    X=X_channel_joint,
    obs=obs,
    obsm=obsm,
)
adata_base

### Computing PCs and neighbors.

In [ ]:
sc.pp.pca(adata_base)
sc.pp.neighbors(adata_base)
sc.tl.umap(adata_base)
adata_base

### Computing neighbor graph.

In [ ]:
k = 30_000
nbrs = NearestNeighbors(n_neighbors=k).fit(adata_base.obsm[cell_state_rep])

## Sample from the prior.

### Generating samples.

In [ ]:
num_prior_samples = 150
num_time_steps = 500
solver_kwargs = {"method":"euler"}
prior_samples = prior_flow.predict(
    {},
    batch_size=num_prior_samples,
    num_time_steps=num_time_steps,
    solver_kwargs=solver_kwargs,
).detach().cpu().numpy()
prior_samples = np.maximum(prior_samples, 0)
print(prior_samples.shape)

### Subsampling train data for efficiency.

In [ ]:
n_train_samples_to_retain = 100_000
train_adata_subset = sc.pp.sample(train_adata, n=n_train_samples_to_retain, copy=True)
train_adata_subset

### Visualize prior samples.

In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(15, 7.5), dpi=100)
# fig, ax = plt.subplots(1, 1, figsize=(15, 7.5), dpi=100)
# fig.suptitle("Heatmap of condition samples")
# plt.figure(figsize=(7.5, 7.5), dpi=100)
# plt.title("Heatmap of condition samples")
sns.clustermap(
    prior_samples,
    annot=False,
    xticklabels=train_adata.var_names,
    # ax=ax[0]
)#; ax[0].tick_params("x", rotation=90, size=4); ax[0].set_title("Prior Samples."); ax[0].set_yticks([])
# plt.title("Heatmap of condition samples")

# ax[0].set_xlabel("Molecule"); ax[0].set_ylabel("Samples")
plt.figure(figsize=(7.5, 7.5), dpi=50)
sns.heatmap(
    train_adata_subset.X,
    annot=False,
    xticklabels=train_adata.var_names,
    # ax=ax[1]
)#; ax[1].tick_params("x", rotation=90, size=4); ax[1].set_title("Subsampled data."); ax[1].set_yticks([])
# ax[1].set_xlabel("Molecule"); ax[1].set_ylabel("Samples")
# fig.tight_layout(); fig.show()

### Plotting the mean and standard deviation over each axes.

In [ ]:
# mean
prior_mean = prior_samples.mean(0)
real_mean = train_adata_subset.X.mean(0)

# standard deviation
prior_std = prior_samples.std(0)
real_std = train_adata_subset.X.std(0)

# initializing plot
fig, ax = plt.subplots(1, 2, figsize=(10, 5), dpi=100)
fig.suptitle("Comparing summary statistics of Learnt Prior distribution to Real Data.")

# contructing linear space
linspace = np.linspace(0.0, 6.5, 100)

# plotting mean
ax[0].scatter(prior_mean, real_mean); ax[0].grid(True)
ax[0].set_title("Mean"); ax[0].set_xlabel("Generated"); ax[0].set_ylabel("Real")
ax[0].plot(linspace, linspace)
yoffset = 0.0
xoffset = 0.3
for i, txt in enumerate(train_adata.var_names):
    ax[0].annotate(txt, (prior_mean[i] + xoffset, real_mean[i] + yoffset), size=6)

# plotting standard deviation
ax[1].scatter(prior_std, real_std); ax[1].grid(True)
ax[1].set_title("Standard Deviation"); ax[1].set_xlabel("Generated"); ax[1].set_ylabel("Real")
for i, txt in enumerate(train_adata.var_names):
    ax[1].annotate(txt, (prior_std[i] + xoffset, real_std[i] + yoffset), size=6)
ax[1].plot(linspace, linspace)

fig.tight_layout(); fig.show()

## Prior Predictive Checks.

### Query forward model with prior samples.

In [ ]:
# n_fwd_samples = 5_000

# # forward pass
# pred_dict = forward_model.predict(
#     {
#         DataFields.PERTURBATION_DATA: {
#             perturbation_reps: torch.from_numpy(prior_samples).float().to(perturbation_response_prediction_model.device)
#         }
#     },
#     num_samples=n_fwd_samples
# )
# X_hat = pred_dict.pop("predicted_states").detach().cpu().numpy()
# G_logits = pred_dict.pop("target_prediction_data")["cell_type"].detach().cpu().numpy()

# # splitting channel and scatter features
# X_channel = X_hat[:, :, :-n_scatter_feats]
# X_scatter = X_hat[:, :, -n_scatter_feats:]
# print(f"{X_hat.shape=}, {X_channel.shape=}, {X_scatter.shape=}")

### Retrieving NN (in subsampled data) for each prior sample.

In [ ]:
# perturbation_id = "condition_concat"
# unique_protocol_conditions = np.unique(
#     train_adata_fwd.obsm[perturbation_id], axis=0
# )
# D = cdist(prior_samples, unique_protocol_conditions)
# nn_idxs = D.argmin(1)
# nn_vals = np.empty_like(prior_samples)
# for idx in range(num_prior_samples):
#     nn_vals[idx] = unique_protocol_conditions[nn_idxs[idx]]
# print(f"{unique_protocol_conditions.shape=}, {D.shape=}, {nn_vals.shape=}")

### Visualizing NN from subsampled adata.

In [ ]:
# fig, ax = plt.subplots(figsize=(5, 5), dpi=100)
# fig.suptitle("NN from training data for each prior samples.", size=10)
# sns.heatmap(nn_vals, annot=False, ax=ax, xticklabels=train_adata.var_names)
# ax.set_yticks([]); ax.set_yticks([])
# ax.set_ylabel("Real Samples"); ax.set_ylabel("Prior Samples")
# fig.tight_layout(); fig.show()

### Visualizing distances.

In [ ]:
# fig, ax = plt.subplots(figsize=(7, 5), dpi=100)
# fig.suptitle("L2 Distance between Real and Generated Samples.", size=10)
# sns.heatmap(D, annot=False, ax=ax)
# ax.set_xticks([]); ax.set_yticks([])
# ax.set_xlabel("Real Samples"); ax.set_ylabel("Prior Samples")
# fig.tight_layout(); fig.show()

### Retrieving prior samples with minimal NN distances.

In [ ]:
# # defining the number of prior samples for which we will
# # visualize the queries from the forward model
# num_prior_samples_to_visualize = 3

# # choosing the samples with the minimum distance to the NN 
# # in the real data and retrieving the corresponding real protocol
# D_min = D.min(1)
# min_idxs_all = np.argsort(D_min)
# min_idxs = min_idxs_all[:num_prior_samples_to_visualize]

### Plotting the UMAP of the predicted subpopulations.

In [ ]:
# # storing the results
# cond2adata = {}

# # iterating over the closest samples 
# for idx in min_idxs:
#     # slicing for current condition
#     sample = prior_samples[idx]

#     x_channel = X_channel[:, idx]
#     x_scatter = X_scatter[:, idx]
#     nn_val = nn_vals[idx]
#     cond2adata[idx] = get_prediction_adata(
#         idx,
#         sample,
#         x_channel,
#         x_scatter,
#         train_adata_fwd,
#         val_adata_fwd,
#         perturbation_id,
#         nn_val,
#         perturbation_obs_key,
#         cell_state_rep,
#         perturbation_reps,
#     )

### Visualizing the sampled protocols.

In [ ]:
# for idx in cond2adata.keys():
#     sample = prior_samples[idx][None]
#     nn_val = nn_vals[idx][None]
#     fig, ax = plt.subplots(2, 1, figsize=(20, 8), dpi=100)
#     sns.heatmap(
#         sample,
#         annot=True,
#         fmt=".2f",
#         ax=ax[0],
#         xticklabels=train_adata.var_names,
#     )
#     ax[0].set_title("Prior Sample")
#     ax[0].set_yticklabels([])

#     sns.heatmap(
#         nn_val,
#         annot=True,
#         fmt=".2f",
#         ax=ax[1],
#         xticklabels=train_adata.var_names,
#     )
#     ax[1].set_title("Nearest Neighbor")
#     ax[1].set_yticklabels([])
#     fig.tight_layout()
#     fig.show()

### Plotting the cells and highlighting generated ones.

In [ ]:
# base_size = 30
# highlight_size = 180
# base_alpha=1.0
# highlight_alpha=1.0
# for idx, adata in cond2adata.items():
#     sc.pl.embedding(adata, "umap", color=["is_generated", "is_true_nn_cond"])


## Inverse Model.

### Initializing guided flow.

In [ ]:
N_FORWARD_PASS_PER_SAMPLE = 500
REGULARIZATION = None
REG_STRENGTH = 1e-2
loss_fns = {"cell_type":lambda pred, target:-torch.sum(target*torch.nn.functional.log_softmax(pred, dim=-1), dim=-1)}
guided_flow = LossGuidedFlow(
    prior_flow,
    forward_model,
    loss_fns,
    fix_noise=True,
    num_forward_pass_per_sample=N_FORWARD_PASS_PER_SAMPLE,
    regularization=REGULARIZATION,
    reg_strength=REG_STRENGTH,
)
non_linearity = torch.nn.ReLU()

### Initializing lambda scheduler.

In [ ]:
LMAX = 10.0
LMIN = 1.0
GAMMA = 1

lambda_scheduler = ExponentialDecayScheduler(LMIN, LMAX, GAMMA)

# visualizing decaying scheduler
t = torch.linspace(0.0, 1.0, 100)
lambda_t = lambda_scheduler.compute_lambda_t(t)
plt.figure(figsize=(3, 3))
plt.plot(t, lambda_t)
plt.title(fr"Decay schedule $\gamma=${GAMMA}")
plt.xlabel(r"$t$")
plt.ylabel(r"$\lambda_t$")
plt.grid(True)



### Preparing configurations for posterior sampling.

In [ ]:
SDE_SAMPLING = False
N_POSTERIOR_SAMPLES = 150
N_TIME_STEPS_POSTERIOR_MODEL = 30 if SDE_SAMPLING else 100
SOLVER_KWARGS_POSTERIOR_MODEL = {"method": "euler"}
seed = 42
set_reproducibility(seed)

### Sampling from posterior.

In [ ]:
# traj, loss_history, lambda_history, noise = guided_flow.sample_posterior(
#     N_POSTERIOR_SAMPLES,
#     optimal_condition={
#         "cell_type":torch.from_numpy(target_probs).unsqueeze(0).to(guided_flow.prior_flow.device)
#     },
#     num_time_steps=N_TIME_STEPS_POSTERIOR_MODEL,
#     solver_kwargs=SOLVER_KWARGS_POSTERIOR_MODEL,
#     lambda_scheduler=lambda_scheduler,
#     non_linearity=non_linearity,
#     sde_sampling=SDE_SAMPLING
# )
# if noise is not None:
#     print(f"{traj.shape=}, {loss_history.shape=}, {lambda_history.shape=}, {noise.shape=}")
# else:
#     print(f"{traj.shape=}, {loss_history.shape=}, {lambda_history.shape=}")

### Plotting Loss History.

In [ ]:
# fig, ax = plt.subplots(figsize=(10, 10), dpi=100)
# t = np.arange(0, 1, 1/loss_history.shape[1])
# lines = ax.plot(t, loss_history.T)
# fig.suptitle("Loss through sampling time.")
# ax.set_xlabel("Sampling Time.")
# ax.set_ylabel("Loss.")
# ax.grid(1)
# fig.tight_layout(); fig.show()

### Comparing with ood protocol.

In [ ]:
# sns.set_style("white")
# sns.set_context("talk")  # poster-friendly font sizes

# estar = np.unique(val_adata_fwd.obsm[perturbation_id], axis=0)

# samples = np.maximum(traj[:, -1, :], 0)
# emin_loss_idx = np.argmin(loss_history[:, -1])
# min_samples = samples[emin_loss_idx][None]

# fig, ax = plt.subplots(1, 3, figsize=(42, 10))  # slightly bigger for poster

# xticks = train_adata.var_names

# # Plot 1: samples (no annotations)
# sns.heatmap(
#     samples,
#     annot=False,
#     cbar=False,
#     xticklabels=xticks,
#     yticklabels=False,
#     ax=ax[0]
# ); ax[0].set_title("Samples")

# # Plot 2: target (with annotations)
# sns.heatmap(
#     estar,
#     annot=True,
#     fmt=".2f",
#     cbar=False,
#     xticklabels=xticks,
#     yticklabels=False,
#     ax=ax[1]
# ); ax[1].set_title("Target")

# # Plot 2: target (with annotations)
# sns.heatmap(
#     min_samples,
#     annot=True,
#     fmt=".2f",
#     cbar=False,
#     xticklabels=xticks,
#     yticklabels=False,
#     ax=ax[2]
# ); ax[2].set_title("Best Sample")

# # Format x-axis labels
# for a in ax:
#     a.tick_params(axis='x', labelsize=16)
#     a.set_xticklabels(a.get_xticklabels(), rotation=90, ha="right")

# fig.tight_layout(pad=2)
# plt.show()

### Querting forward model.

In [ ]:
# num_time_steps = 500
# solver_kwargs = {"method": "euler"}
# ccondition_data = torch.from_numpy(samples).float().to(perturbation_response_prediction_model.device)
# ccondition_dict = {
#     perturbation_reps: ccondition_data.unsqueeze(1).repeat(1, N_FORWARD_PASS_PER_SAMPLE, 1),
# }
# cforward_out = forward_model.predict(
#     {
#         DataFields.SOURCE_STATE: noise,
#         DataFields.PERTURBATION_DATA: ccondition_dict
#     },
#     return_trajectory=False,
#     no_grad=True,
#     num_time_steps=num_time_steps,
#     solver_kwargs=solver_kwargs,
#     fix_noise=True,
# )
# X_gen = cforward_out[PredictionFields.PREDICTION_DATA].detach().cpu().numpy()
# X_channel_gen = X_gen[:, :, :-n_scatter_feats]
# X_scatter_gen = X_gen[:, :, -n_scatter_feats:]
# gen_region_logits = cforward_out[PredictionFields.TARGET_PREDICTION_DATA]["cell_type"].detach().cpu().numpy()
# gen_region_probs = softmax(gen_region_logits, axis=-1)
# print(f"{samples.shape=} {X_gen.shape=} {gen_region_probs.shape=}, {X_channel_gen.shape=}, {X_scatter_gen.shape=}")


### Displaying induced phenotype.

In [ ]:
# emin_loss_idx = np.argmin(loss_history[:, -1])
# phenotype = gen_region_probs.mean(1)
# best_pheno = gen_region_probs[emin_loss_idx]

# fig, ax = plt.subplots(1, 3, figsize=(42, 10))  # bigger for poster

# xticks = [le_ct.inverse_transform(np.array([i])).item() for i in range(best_pheno.shape[-1])]

# heatmap_kwargs = dict(
#     annot=False,
#     cbar=False,
#     xticklabels=xticks,
#     yticklabels=False,
# )

# sns.heatmap(phenotype, ax=ax[0], **heatmap_kwargs)
# # sns.heatmap(best_pheno, ax=ax[1], **heatmap_kwargs)

# heatmap_kwargs["annot"] = True
# sns.heatmap(best_pheno.mean(0)[None], ax=ax[1], fmt=".2f", **heatmap_kwargs)
# sns.heatmap(target_probs[None], ax=ax[2], fmt=".2f", **heatmap_kwargs)

# # rotate x labels for readability
# for a in ax:
#     a.tick_params(axis='x', labelsize=14)  # larger font
#     a.set_xticklabels(a.get_xticklabels(), rotation=90, ha="right")

# fig.tight_layout(pad=2)
# plt.show()

### Constructing adata.

In [ ]:
# emin_loss_idx = np.argmin(loss_history[:, -1])

# X_channel_gen_emin = X_channel_gen[emin_loss_idx]
# X_scatter_gen_emin = X_scatter_gen[emin_loss_idx]
# gen_region_probs = gen_region_probs[emin_loss_idx]

# X_train = train_adata_fwd.obsm[cell_state_rep]
# X_val = val_adata_fwd.obsm[cell_state_rep]

# X_channel_train = X_train[:, :-n_scatter_feats]
# X_scatter_train = X_train[:, -n_scatter_feats:]

# X_channel_val = X_val[:, :-n_scatter_feats]
# X_scatter_val = X_val[:, -n_scatter_feats:]

# with torch.no_grad():
#     train_region_probs = g_model(torch.from_numpy(X_train).float().to("cuda"))["cell_type"].detach().cpu().numpy()
#     val_region_probs = g_model(torch.from_numpy(X_val).float().to("cuda"))["cell_type"].detach().cpu().numpy()

# X_channel = np.concatenate((X_channel_train, X_channel_val, X_channel_gen_emin), axis=0)
# X_scatter = np.concatenate((X_scatter_train, X_scatter_val, X_scatter_gen_emin), axis=0)
# # Y = np.concatenate((train_region_probs, val_region_probs, gen_region_probs), axis=0)
# obs={
#     "dataset_type": ["train" for _ in range(X_train.shape[0])] + \
#         ["ood" for _ in range(X_channel_val.shape[0])] + \
#             ["generated" for _ in range(N_FORWARD_PASS_PER_SAMPLE)],
# }
# obsm = {
#     "X_channel": X_channel,
#     "X_scatter": X_scatter,
# }

# adata_gen = sc.AnnData(
#     X_channel,
#     obsm=obsm,
#     obs=obs
# )
# adata_gen.obs["is_generated"] = adata_gen.obs["dataset_type"] == "generated"

# sc.pp.pca(adata_gen)
# sc.pp.neighbors(adata_gen)
# sc.tl.umap(adata_gen)

### Visualizing cell states.

In [ ]:
# # defining figure
# fig, ax = plt.subplots(1, 3, figsize=(15, 5))
# X_umap = adata_gen.obsm["X_umap"]

# for idx, unique_val in enumerate(adata_gen.obs["dataset_type"].unique()):
#     mask =  adata_gen.obs["dataset_type"] == unique_val
#     cax = ax[idx]
#     # plotting base coords
#     cax.set_title(f"{idx}:{unique_val}")
#     cax.scatter(
#         X_umap[:, 0],
#         X_umap[:, 1],
#         c='lightgrey',  # background cells
#         s=20,           # smaller size
#         alpha=0.5
#     )
#     cax.scatter(
#         X_umap[mask, 0],
#         X_umap[mask, 1],
#         alpha=0.5,
#         s=80,
#         edgecolor='k',
#     )

## Querying pure populations.

### Calling the inverse model for each cell type.

In [ ]:
results_dict = {}
for idx, ct in enumerate(le_ct.classes_):
    if ct != "MgkPro":
        continue
    print(ct)
    g_star = np.zeros_like(target_probs)
    g_star[idx] = 1.0
    traj, loss_history, lambda_history, noise = guided_flow.sample_posterior(
        N_POSTERIOR_SAMPLES,
        optimal_condition={
            "cell_type":torch.from_numpy(g_star).unsqueeze(0).to(guided_flow.prior_flow.device)
        },
        num_time_steps=N_TIME_STEPS_POSTERIOR_MODEL,
        solver_kwargs=SOLVER_KWARGS_POSTERIOR_MODEL,
        lambda_scheduler=lambda_scheduler,
        non_linearity=non_linearity,
        sde_sampling=SDE_SAMPLING
    )
    results_dict[ct] = {
        "traj": traj,
        "loss_history": loss_history,
        "lambda_history": lambda_history,
        "noise": noise
    }


### Saving results dict.

In [ ]:
with open("../posterior_results.pkl", "wb") as fb:
    cloudpickle.dump(results_dict, fb)

### Loading results dict.

In [ ]:
with open("../posterior_results.pkl", "rb") as fb:
    results_dict = cloudpickle.load(fb)

### Plotting the losses for each queried cell type.

In [ ]:
for ct, res_dict in results_dict.items():
    loss_history = res_dict["loss_history"]
    fig, ax = plt.subplots(figsize=(7, 7), dpi=50)
    t = np.arange(0, 1, 1/loss_history.shape[1])
    lines = ax.plot(t, loss_history.T)
    fig.suptitle(f"{ct}")
    ax.set_xlabel("Sampling Time.")
    ax.set_ylabel("Loss.")
    ax.grid(1)
    fig.tight_layout(); fig.show()
    safe_ct = ct.replace("/", "_")
    fig.savefig(f"../plots/losses_{safe_ct}.png")

### Querying forward model.

In [ ]:
N_FORWARD_PASS_PER_SAMPLE = 500
num_time_steps = 500
solver_kwargs = {"method": "euler"}

ct_preds = {}
for ct, res_dict in results_dict.items():
    # query forward model
    print(ct)
    traj = res_dict["traj"]
    noise = res_dict["noise"]
    samples = np.maximum(traj[:, -1, :], 0)
    ccondition_data = torch.from_numpy(samples).float().to(perturbation_response_prediction_model.device)
    ccondition_dict = {
        perturbation_reps: ccondition_data.unsqueeze(1).repeat(1, N_FORWARD_PASS_PER_SAMPLE, 1),
    }
    batch_dict = {
        DataFields.PERTURBATION_DATA: ccondition_dict,
        DataFields.SOURCE_STATE: noise
    }
    cforward_out = forward_model.predict(
        batch_dict,
        return_trajectory=False,
        no_grad=True,
        # num_samples=n_samples,
        num_time_steps=num_time_steps,
        solver_kwargs=solver_kwargs,
        fix_noise=True,
    )
    X_gen = cforward_out[PredictionFields.PREDICTION_DATA].detach().cpu().numpy()
    gen_ct_logits = cforward_out[PredictionFields.TARGET_PREDICTION_DATA]["cell_type"].detach().cpu().numpy()
    X_channel_gen = X_gen[..., :-n_scatter_feats]
    X_scatter_gen = X_gen[..., -n_scatter_feats:]
    gen_ct_probs = softmax(gen_ct_logits, axis=-1)
    gen_ct_id_label = gen_ct_probs.argmax(-1)
    gen_ct_label = le_ct.inverse_transform(gen_ct_id_label.reshape(-1)).\
        reshape(gen_ct_id_label.shape[0], gen_ct_id_label.shape[1])
    print(f"{X_gen.shape=}, {X_channel_gen.shape=}, {X_scatter_gen.shape=}, {gen_ct_logits.shape=} {gen_ct_id_label.shape=}, {gen_ct_label.shape=}")
    ct_preds[ct] = {
        "X": X_gen,
        "X_channel": X_channel_gen,
        "X_scatter": X_scatter_gen,
        "ct_logits": gen_ct_logits,
        "ct_probs": gen_ct_probs,
        "ct_label": gen_ct_label,
    }
with open("../posterior_fwd_queries.pkl", "wb") as fb:
    cloudpickle.dump(ct_preds, fb)


### Displaying samples.

In [ ]:
for ct, preds_dict in ct_preds.items():
    loss_history = results_dict[ct]["loss_history"]
    emin_loss_idx = np.argmin(loss_history[:, -1])
    ct_probs = preds_dict["ct_probs"].mean(1)
    traj = res_dict["traj"]
    samples = np.maximum(traj[:, -1, :], 0)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4), dpi=100)
    fig.suptitle(ct)
    sns.heatmap(samples, annot=False, ax=ax[0], xticklabels=annotation_dict["protocol_columns"])
    sns.heatmap(samples[emin_loss_idx][None], annot=True, ax=ax[1], xticklabels=annotation_dict["protocol_columns"], fmt=".2f", annot_kws={"size": 6})
    safe_ct = ct.replace("/", "_")
    fig.savefig(f"../plots/samples_{safe_ct}.png")

### Displaying induced phenotypes.

In [ ]:
for ct, preds_dict in ct_preds.items():
    loss_history = results_dict[ct]["loss_history"]
    emin_loss_idx = np.argmin(loss_history[:, -1])
    ct_probs = preds_dict["ct_probs"].mean(1)
    traj = res_dict["traj"]
    samples = np.maximum(traj[:, -1, :], 0)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4), dpi=100)
    fig.suptitle(ct)
    sns.heatmap(ct_probs, annot=False, ax=ax[0], xticklabels=ct_le.classes_)
    sns.heatmap(ct_probs[emin_loss_idx][None], annot=True, ax=ax[1], xticklabels=ct_le.classes_, fmt=".2f", annot_kws={"size": 6})
    safe_ct = ct.replace("/", "_")
    fig.savefig(f"../plots/induced_pheno_{safe_ct}.png")

### Creating concatenated data.

In [ ]:
adatas_dict = {}
for ct, preds_dict in ct_preds.items():
    print(ct)
    loss_history = results_dict[ct]["loss_history"]
    emin_loss_idx = np.argmin(loss_history[:, -1])
    X = np.concatenate((preds_dict["X"][emin_loss_idx],
        train_adata_fwd.obsm[cell_state_rep],
        val_adata_fwd.obsm[cell_state_rep]), axis=0)
    g_model.eval()
    with torch.no_grad():
        g_logits = g_model(torch.from_numpy(X).cuda())["cell_type"].detach().cpu().numpy()
    g_probs = softmax(g_logits, axis=-1)
    g_label = ct_le.inverse_transform(g_probs.argmax(1))
    X_channel = X[:, :-n_scatter_feats]
    X_scatter = X[:, -n_scatter_feats:]
    ct_adata = sc.AnnData(
        X=X_channel,
        obsm={"X_scatter": X_scatter},
        obs={
            "cell_type": g_label,
            "data_type": ["gen"]*preds_dict["X"][emin_loss_idx].shape[0] + \
                ["train"]*len(train_adata_fwd) + ["val"]*len(val_adata_fwd)
        },
        var=pd.DataFrame(index=train_adata_fwd.var_names)
    )
    sc.pp.pca(ct_adata)
    sc.pp.neighbors(ct_adata)
    sc.tl.umap(ct_adata)
    adatas_dict[ct] = ct_adata


### Plotting UMAP with concatenated anndata.

In [ ]:
for ct, ct_adata in adatas_dict.items():
    X_umap = ct_adata.obsm["X_umap"]
    ct_mask = (ct_adata.obs["cell_type"] == ct) & (ct_adata.obs["data_type"] != "gen")
    gen_mask = ct_adata.obs["data_type"] == "gen"
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
    fig.suptitle(ct)
    axes[0].scatter(
        X_umap[:, 0],
        X_umap[:, 1],
        s=5,
        # alpha=0.2
    )
    axes[0].scatter(
        X_umap[ct_mask, 0],
        X_umap[ct_mask, 1],
        s=40,
        # alpha=0.9
    )
    axes[0].set_title(f"Is Target")
    axes[0].grid(1)

    axes[1].scatter(
        X_umap[:, 0],
        X_umap[:, 1],
        s=5,
        # alpha=0.2
    )
    axes[1].scatter(
        X_umap[gen_mask, 0],
        X_umap[gen_mask, 1],
        s=40,
        # alpha=0.9
    )
    axes[1].set_title("Is Generated")
    axes[1].grid(1)

    for ax in axes:
        ax.set_xlabel("UMAP1")
        ax.set_ylabel("UMAP2")

    plt.tight_layout()
    plt.show()
    safe_ct = ct.replace("/", "_")
    fig.savefig(f"../plots/cell_states_umap_{safe_ct}.png")

### Plotting marginals.

### Dot plot expression for each cell type for generated cells.

In [ ]:
X_list = []
ct_list = []
for ct, preds_dict in ct_preds.items():
    loss_history = results_dict[ct]["loss_history"]
    emin_loss_idx = np.argmin(loss_history[:, -1])
    X = preds_dict["X"][emin_loss_idx]
    X_list.append(X)
    ct_list.extend([ct]*X.shape[0])
X = np.concatenate(X_list, axis=0)
X = izn(torch.from_numpy(X).cuda()).detach().cpu().numpy()
adata_preds = sc.AnnData(
    X=X[:, :-n_scatter_feats],
    obs={"cell_type": ct_list},
    var=pd.DataFrame(index=train_adata_fwd.var_names)
)
sc.pl.dotplot(adata_preds, adata_preds.var_names, "cell_type")
adata_preds

### Dot plot expression for each marker per cell type real data.

In [ ]:
izn_g = IZNorm(params_fwd)
X = np.concatenate(
    (
        target_prediction_model.train_data.adata.obsm[cell_state_rep],
        target_prediction_model.validation_data.adata.obsm[cell_state_rep],
    ), axis=0
)
X = izn_g(torch.from_numpy(X).cuda()).detach().cpu().numpy()
adata_gt = sc.AnnData(
    X=X[:, :-n_scatter_feats],
    obs={"cell_type": target_prediction_model.train_data.adata.obs["cell_type"].tolist() +\
        target_prediction_model.validation_data.adata.obs["cell_type"].tolist()},
    var=pd.DataFrame(index=train_adata_fwd.var_names)
)
sc.pl.dotplot(adata_gt, adata_gt.var_names, "cell_type")
adata_gt